# Bank Statement Transaction Extraction with Llama Vision

Specialized notebook for extracting transactional data from bank statements using Llama-3.2-Vision-Instruct.

**Key Features:**
- Handles empty debit/credit cells correctly
- Preserves row-by-row transaction structure
- Calculates running balances
- Validates transaction integrity

In [ ]:
# Configuration
MODEL_PATH = "/path/to/Llama-3.2-11B-Vision-Instruct"  # UPDATE THIS PATH
STATEMENT_IMAGE_PATH = "path/to/bank_statement.jpg"  # UPDATE THIS PATH

# GPU settings
LOAD_IN_8BIT = True  # Enable 8-bit quantization for V100/memory efficiency
DEVICE_MAP = "auto"  # Automatic device mapping
MAX_NEW_TOKENS = 3000  # Increased for full statement extraction

In [ ]:
# Install required packages if needed
# !pip install transformers torch torchvision pillow accelerate bitsandbytes pandas

In [ ]:
import torch
from transformers import MllamaForConditionalGeneration, AutoProcessor
from PIL import Image
import json
import re
from typing import Dict, List, Any, Optional, Tuple
import pandas as pd
from datetime import datetime
from decimal import Decimal, InvalidOperation
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Initialize model and processor
print("Loading Llama Vision model for bank statement processing...")

model = MllamaForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE_MAP,
    load_in_8bit=LOAD_IN_8BIT if LOAD_IN_8BIT else None,
)

processor = AutoProcessor.from_pretrained(MODEL_PATH)

print("✅ Model loaded successfully")
print(f"📊 Device: {next(model.parameters()).device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name()}")
    print(f"💾 Memory: {torch.cuda.memory_allocated()/1e9:.2f}GB allocated")

In [ ]:
# Specialized bank statement extraction prompts
BANK_STATEMENT_PROMPTS = {
    "structured_transactions": """Analyze this bank statement image and extract ALL transactions as a structured table.

CRITICAL INSTRUCTIONS:
1. Treat the statement as a TABLE with distinct columns
2. Each transaction is ONE ROW in the table
3. Empty cells MUST be marked as null or empty string
4. PRESERVE the exact row structure - do not skip or combine rows

For EACH transaction row, extract:
- Date (transaction date)
- Description (transaction description/details)
- Debit (amount if it's a debit/withdrawal, otherwise null)
- Credit (amount if it's a credit/deposit, otherwise null)  
- Balance (running balance after this transaction)

Return as JSON:
{
  "account_details": {
    "account_number": "...",
    "statement_period": "...",
    "opening_balance": "...",
    "closing_balance": "..."
  },
  "transactions": [
    {
      "date": "DD/MM/YYYY",
      "description": "...",
      "debit": "100.00" or null,
      "credit": null or "200.00",
      "balance": "1234.56"
    }
  ]
}

IMPORTANT: A transaction will have EITHER a debit OR a credit value, never both. The other MUST be null.""",

    "row_by_row_extraction": """Extract the transaction table from this bank statement image ROW BY ROW.

CRITICAL: This is a TABLE where each row represents one transaction.

For the transaction table:
1. Identify ALL column headers (typically: Date, Description, Debit, Credit, Balance)
2. For EACH row in the table, extract ALL cells
3. If a cell is EMPTY (like debit column for a credit transaction), mark it as "EMPTY"
4. Preserve EXACT row order - do not skip any rows

Format your response EXACTLY as:
COLUMNS: Date | Description | Debit | Credit | Balance
ROW 1: [date] | [description] | [amount or EMPTY] | [amount or EMPTY] | [balance]
ROW 2: [date] | [description] | [amount or EMPTY] | [amount or EMPTY] | [balance]
[continue for all rows]

Remember: Each transaction has EITHER debit OR credit, so one will always be EMPTY.""",

    "csv_with_empty_cells": """Convert the bank statement transaction table to CSV format.

RULES FOR BANK STATEMENTS:
1. First line: Date,Description,Debit,Credit,Balance
2. Each transaction is one line
3. Empty debit/credit cells must be represented as empty (,,)
4. DO NOT skip empty cells - preserve the column structure
5. Amounts should not include currency symbols

Example format:
Date,Description,Debit,Credit,Balance
01/03/2024,Opening Balance,,,5000.00
02/03/2024,ATM Withdrawal,100.00,,4900.00
03/03/2024,Salary Deposit,,3500.00,8400.00

Extract ALL transactions maintaining this exact structure.""",

    "detailed_with_metadata": """Carefully analyze this bank statement and extract all information.

Extract:
1. HEADER INFORMATION:
   - Bank name
   - Account holder name
   - Account number (masked is ok)
   - Statement period (from date to date)
   - Opening balance
   - Closing balance

2. TRANSACTION TABLE (treat as structured table):
   For each row in the transaction table:
   - Date
   - Transaction description/particulars
   - Debit amount (if present, otherwise "nil")
   - Credit amount (if present, otherwise "nil")
   - Running balance

3. SUMMARY (if present):
   - Total debits
   - Total credits
   - Number of transactions

CRITICAL: Preserve the table structure. Each transaction occupies exactly one row.
Mark empty cells explicitly as "nil" or "empty".""",

    "validation_focused": """Extract bank statement transactions with validation checks.

For each transaction in the table:
1. Date (format: DD/MM/YYYY or MM/DD/YYYY)
2. Description (full text)
3. Debit (negative/outgoing amount or NULL)
4. Credit (positive/incoming amount or NULL)
5. Balance (running balance)

VALIDATION RULES:
- Each row must have EITHER debit OR credit, not both
- Balance should equal: previous_balance - debit + credit
- All amounts must be positive numbers (no negative signs)
- Dates must be in chronological order

Return as JSON with a 'validation' field indicating any issues found."""
}

In [ ]:
def extract_bank_statement(
    image_path: str, 
    prompt_type: str = "structured_transactions",
    custom_prompt: str = None,
    temperature: float = 0.1
) -> Dict[str, Any]:
    """
    Extract bank statement data from an image.
    
    Args:
        image_path: Path to bank statement image
        prompt_type: Type of extraction prompt
        custom_prompt: Optional custom prompt
        temperature: Generation temperature (lower = more deterministic)
    
    Returns:
        Dictionary with extraction results
    """
    # Load image
    image = Image.open(image_path)
    
    # Select prompt
    prompt = custom_prompt if custom_prompt else BANK_STATEMENT_PROMPTS.get(
        prompt_type, BANK_STATEMENT_PROMPTS["structured_transactions"]
    )
    
    # Prepare inputs
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]}
    ]
    
    # Apply chat template
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    
    # Process inputs
    inputs = processor(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)
    
    # Generate response
    print("🔍 Analyzing bank statement...")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=temperature,
            top_p=0.95
        )
    
    # Decode response
    response = processor.decode(output[0], skip_special_tokens=True)
    
    # Extract assistant response
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    
    return {
        "raw_response": response,
        "prompt_type": prompt_type,
        "image_path": image_path,
        "extraction_time": datetime.now().isoformat()
    }

In [ ]:
def parse_statement_json(response: str) -> Optional[Dict]:
    """
    Parse JSON response from bank statement extraction.
    """
    try:
        # Find JSON in response
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group())
            return data
    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing error: {e}")
    return None

def parse_row_format(response: str) -> pd.DataFrame:
    """
    Parse row-by-row format response into DataFrame.
    """
    lines = response.strip().split('\n')
    
    # Find column headers
    columns = None
    rows = []
    
    for line in lines:
        if line.startswith('COLUMNS:'):
            columns = [col.strip() for col in line.replace('COLUMNS:', '').split('|')]
        elif line.startswith('ROW'):
            # Extract row data
            row_data = line.split(':', 1)[1] if ':' in line else line
            values = [val.strip() for val in row_data.split('|')]
            
            # Replace EMPTY with None
            values = [None if v.upper() == 'EMPTY' else v for v in values]
            rows.append(values)
    
    if columns and rows:
        return pd.DataFrame(rows, columns=columns)
    return None

def parse_csv_format(response: str) -> pd.DataFrame:
    """
    Parse CSV format with proper handling of empty cells.
    """
    try:
        lines = response.strip().split('\n')
        csv_lines = []
        
        for line in lines:
            # Only include lines that look like CSV
            if ',' in line and not line.startswith('#'):
                csv_lines.append(line)
        
        if csv_lines:
            from io import StringIO
            csv_text = '\n'.join(csv_lines)
            df = pd.read_csv(StringIO(csv_text))
            
            # Replace NaN with None for clarity
            df = df.where(pd.notnull(df), None)
            return df
    except Exception as e:
        print(f"❌ CSV parsing error: {e}")
    return None

In [ ]:
def validate_transactions(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Validate bank statement transactions.
    
    Returns:
        Dictionary with validation results and issues
    """
    issues = []
    
    # Check for required columns
    required_cols = ['date', 'description', 'debit', 'credit', 'balance']
    df_cols_lower = [col.lower() for col in df.columns]
    
    for req_col in required_cols:
        if req_col not in df_cols_lower:
            issues.append(f"Missing required column: {req_col}")
    
    # Normalize column names
    df.columns = [col.lower() for col in df.columns]
    
    # Validation checks
    for idx, row in df.iterrows():
        # Check debit/credit exclusivity
        has_debit = row.get('debit') not in [None, '', 'nil', 'empty', 'EMPTY']
        has_credit = row.get('credit') not in [None, '', 'nil', 'empty', 'EMPTY']
        
        if has_debit and has_credit:
            issues.append(f"Row {idx}: Has both debit and credit values")
        elif not has_debit and not has_credit:
            # Check if it's not an opening/closing balance row
            desc = str(row.get('description', '')).lower()
            if 'opening' not in desc and 'closing' not in desc:
                issues.append(f"Row {idx}: Missing both debit and credit values")
    
    # Calculate totals
    total_debits = 0
    total_credits = 0
    
    for _, row in df.iterrows():
        try:
            debit_val = row.get('debit')
            if debit_val and str(debit_val).replace('.', '').replace(',', '').isdigit():
                total_debits += float(str(debit_val).replace(',', ''))
            
            credit_val = row.get('credit')
            if credit_val and str(credit_val).replace('.', '').replace(',', '').isdigit():
                total_credits += float(str(credit_val).replace(',', ''))
        except:
            pass
    
    return {
        "valid": len(issues) == 0,
        "issues": issues,
        "total_debits": total_debits,
        "total_credits": total_credits,
        "net_change": total_credits - total_debits,
        "transaction_count": len(df)
    }

In [ ]:
def calculate_running_balance(df: pd.DataFrame, opening_balance: float = None) -> pd.DataFrame:
    """
    Calculate or verify running balance for transactions.
    """
    df = df.copy()
    
    # Normalize column names
    df.columns = [col.lower() for col in df.columns]
    
    # Convert amounts to float
    def to_float(val):
        if val in [None, '', 'nil', 'empty', 'EMPTY', 'null']:
            return 0.0
        try:
            return float(str(val).replace(',', '').replace('$', ''))
        except:
            return 0.0
    
    df['debit_amount'] = df['debit'].apply(to_float)
    df['credit_amount'] = df['credit'].apply(to_float)
    df['balance_amount'] = df['balance'].apply(to_float)
    
    # Calculate expected balance
    if opening_balance is None and len(df) > 0:
        # Try to infer opening balance from first row
        first_balance = df.iloc[0]['balance_amount']
        first_debit = df.iloc[0]['debit_amount']
        first_credit = df.iloc[0]['credit_amount']
        opening_balance = first_balance + first_debit - first_credit
    
    running_balance = opening_balance if opening_balance else 0
    df['calculated_balance'] = 0.0
    df['balance_match'] = True
    
    for idx, row in df.iterrows():
        running_balance = running_balance - row['debit_amount'] + row['credit_amount']
        df.at[idx, 'calculated_balance'] = running_balance
        
        # Check if balances match (within tolerance for rounding)
        if row['balance_amount'] > 0:
            diff = abs(running_balance - row['balance_amount'])
            df.at[idx, 'balance_match'] = diff < 0.01
    
    return df

## Extract Bank Statement Transactions

In [ ]:
# Extract using structured JSON format
print("📄 Extracting bank statement transactions...")
print("=" * 60)

result = extract_bank_statement(
    STATEMENT_IMAGE_PATH, 
    prompt_type="structured_transactions"
)

print("\n📊 Raw Response (first 1000 chars):")
print("-" * 60)
print(result["raw_response"][:1000])
print("..." if len(result["raw_response"]) > 1000 else "")
print("-" * 60)

In [ ]:
# Parse and display structured data
parsed_data = parse_statement_json(result["raw_response"])

if parsed_data:
    print("✅ Successfully parsed bank statement data\n")
    
    # Display account details
    if "account_details" in parsed_data:
        print("🏦 Account Details:")
        for key, value in parsed_data["account_details"].items():
            print(f"  {key.replace('_', ' ').title()}: {value}")
    
    # Convert transactions to DataFrame
    if "transactions" in parsed_data:
        transactions_df = pd.DataFrame(parsed_data["transactions"])
        
        print(f"\n📊 Transactions: {len(transactions_df)} found")
        print("\nFirst 5 transactions:")
        display(transactions_df.head())
        
        # Save to CSV
        output_file = "extracted_bank_transactions.csv"
        transactions_df.to_csv(output_file, index=False)
        print(f"\n💾 Saved to {output_file}")
else:
    print("⚠️ Could not parse JSON. Trying row-by-row format...")

In [ ]:
# Try row-by-row extraction for better structure preservation
print("\n🔍 Trying row-by-row extraction...")
row_result = extract_bank_statement(
    STATEMENT_IMAGE_PATH,
    prompt_type="row_by_row_extraction"
)

print("\n📊 Row Format Response (first 1500 chars):")
print("-" * 60)
print(row_result["raw_response"][:1500])
print("-" * 60)

# Parse row format
row_df = parse_row_format(row_result["raw_response"])
if row_df is not None:
    print(f"\n✅ Extracted {len(row_df)} transaction rows")
    print("\nTransaction Table:")
    display(row_df)
    
    # Use this DataFrame for further processing
    transactions_df = row_df

In [ ]:
# Validate transactions
if 'transactions_df' in locals() and transactions_df is not None:
    print("\n🔍 Validating Transactions...")
    print("=" * 60)
    
    validation_results = validate_transactions(transactions_df)
    
    if validation_results["valid"]:
        print("✅ All transactions passed validation!")
    else:
        print("⚠️ Validation issues found:")
        for issue in validation_results["issues"]:
            print(f"  - {issue}")
    
    print(f"\n📊 Transaction Summary:")
    print(f"  Total Transactions: {validation_results['transaction_count']}")
    print(f"  Total Debits: ${validation_results['total_debits']:,.2f}")
    print(f"  Total Credits: ${validation_results['total_credits']:,.2f}")
    print(f"  Net Change: ${validation_results['net_change']:,.2f}")

In [ ]:
# Calculate and verify running balance
if 'transactions_df' in locals() and transactions_df is not None:
    print("\n💰 Calculating Running Balance...")
    print("=" * 60)
    
    # Calculate with balance verification
    balanced_df = calculate_running_balance(transactions_df)
    
    # Show results
    print("\nTransactions with Balance Verification:")
    display_cols = ['date', 'description', 'debit', 'credit', 'balance', 'calculated_balance', 'balance_match']
    display_cols = [col for col in display_cols if col in balanced_df.columns]
    display(balanced_df[display_cols].head(10))
    
    # Check for mismatches
    mismatches = balanced_df[balanced_df['balance_match'] == False]
    if len(mismatches) > 0:
        print(f"\n⚠️ Found {len(mismatches)} balance mismatches:")
        display(mismatches[display_cols])
    else:
        print("\n✅ All balances match calculated values!")
    
    # Save enhanced version
    balanced_df.to_csv("bank_transactions_with_validation.csv", index=False)
    print("\n💾 Saved validated transactions to bank_transactions_with_validation.csv")

In [ ]:
# Advanced: Extract with custom prompt for specific bank formats
custom_bank_prompt = """
Extract the transaction table from this bank statement.

CRITICAL: This is a TABULAR structure where:
- Each row represents exactly ONE transaction
- Debit column shows withdrawals/payments (empty for deposits)
- Credit column shows deposits/receipts (empty for withdrawals)
- One of debit/credit MUST be empty for each transaction

For each transaction row, extract these columns IN ORDER:
1. Date (as shown)
2. Transaction details/description
3. Reference number (if present, else "N/A")
4. Debit amount (if present, else "0.00")
5. Credit amount (if present, else "0.00")  
6. Balance

Format as pipe-separated values:
Date|Description|Reference|Debit|Credit|Balance
[actual data rows follow]

Preserve EXACT row structure - do not combine or skip rows.
"""

print("\n🎯 Using custom extraction prompt...")
custom_result = extract_bank_statement(
    STATEMENT_IMAGE_PATH,
    custom_prompt=custom_bank_prompt
)

print("\nCustom extraction result (first 800 chars):")
print(custom_result["raw_response"][:800])

In [ ]:
# Batch processing for multiple statement pages
def process_multi_page_statement(image_paths: List[str]) -> pd.DataFrame:
    """
    Process multiple pages of a bank statement.
    """
    all_transactions = []
    
    for i, image_path in enumerate(image_paths, 1):
        print(f"\n📄 Processing page {i}/{len(image_paths)}...")
        
        try:
            # Extract transactions
            result = extract_bank_statement(image_path, "structured_transactions")
            parsed = parse_statement_json(result["raw_response"])
            
            if parsed and "transactions" in parsed:
                page_df = pd.DataFrame(parsed["transactions"])
                page_df['page'] = i
                all_transactions.append(page_df)
                print(f"  ✅ Extracted {len(page_df)} transactions")
            else:
                print(f"  ⚠️ No transactions found")
                
        except Exception as e:
            print(f"  ❌ Error: {e}")
    
    if all_transactions:
        combined_df = pd.concat(all_transactions, ignore_index=True)
        print(f"\n✅ Total transactions extracted: {len(combined_df)}")
        return combined_df
    
    return pd.DataFrame()

# Example usage (uncomment and update paths):
# statement_pages = ["statement_page1.jpg", "statement_page2.jpg"]
# multi_page_df = process_multi_page_statement(statement_pages)
# multi_page_df.to_csv("complete_statement.csv", index=False)

In [ ]:
# Export to different formats
if 'transactions_df' in locals() and transactions_df is not None:
    print("\n📤 Exporting Transactions...")
    
    # 1. Excel format with formatting
    with pd.ExcelWriter('bank_transactions.xlsx', engine='openpyxl') as writer:
        transactions_df.to_excel(writer, sheet_name='Transactions', index=False)
        
        # Add summary sheet if validation was done
        if 'validation_results' in locals():
            summary_df = pd.DataFrame([{
                'Total Transactions': validation_results['transaction_count'],
                'Total Debits': f"${validation_results['total_debits']:,.2f}",
                'Total Credits': f"${validation_results['total_credits']:,.2f}",
                'Net Change': f"${validation_results['net_change']:,.2f}"
            }])
            summary_df.T.to_excel(writer, sheet_name='Summary')
    
    print("  ✅ Saved to bank_transactions.xlsx")
    
    # 2. JSON format for APIs
    transactions_json = transactions_df.to_json(orient='records', indent=2)
    with open('bank_transactions.json', 'w') as f:
        f.write(transactions_json)
    print("  ✅ Saved to bank_transactions.json")
    
    # 3. Accounting software format (QIF)
    qif_output = "!Type:Bank\n"
    for _, row in transactions_df.iterrows():
        qif_output += f"D{row.get('date', '')}\n"
        qif_output += f"P{row.get('description', '')}\n"
        
        # Amount (negative for debits)
        if row.get('debit') and str(row['debit']).replace('.','').isdigit():
            qif_output += f"T-{row['debit']}\n"
        elif row.get('credit') and str(row['credit']).replace('.','').isdigit():
            qif_output += f"T{row['credit']}\n"
        qif_output += "^\n"
    
    with open('bank_transactions.qif', 'w') as f:
        f.write(qif_output)
    print("  ✅ Saved to bank_transactions.qif (Quicken format)")
    
    print("\n📊 Export complete!")

In [ ]:
# Clean up GPU memory
def cleanup_gpu():
    """
    Clean up GPU memory after processing.
    """
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("🧹 GPU memory cleaned")
        print(f"   Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        print(f"   Reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")

# Uncomment to clean up
# cleanup_gpu()

## Tips for Bank Statement Extraction

### Key Challenges & Solutions

1. **Empty Debit/Credit Cells**
   - The prompts explicitly instruct to mark empty cells
   - Each transaction has EITHER debit OR credit, never both
   - Validation checks ensure this rule is followed

2. **Maintaining Row Structure**
   - Prompts emphasize treating statements as tables
   - Row-by-row extraction preserves exact structure
   - No skipping or combining of rows

3. **Balance Verification**
   - Automatic calculation of expected balances
   - Comparison with extracted balances
   - Highlights any discrepancies

### Best Practices

- **Image Quality**: Higher resolution improves accuracy
- **Clear Tables**: Well-defined borders help extraction
- **Multiple Formats**: Try different prompt types for best results
- **Validation**: Always validate debit/credit exclusivity
- **Multi-Page**: Process each page separately, then combine

### Common Bank Statement Formats

- **Standard**: Date | Description | Debit | Credit | Balance
- **With Reference**: Date | Description | Reference | Debit | Credit | Balance  
- **Combined Amount**: Date | Description | Amount | Balance (use +/- signs)
- **Separate Columns**: Date | Withdrawals | Deposits | Balance

### Export Options

- **CSV**: Universal format for spreadsheets
- **Excel**: With formatting and summary sheets
- **JSON**: For APIs and web applications
- **QIF**: For accounting software import